In [2]:
import os
from dotenv import load_dotenv
from pydantic import create_model
import inspect, json
from inspect import Parameter

print(load_dotenv())

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

True


## **Custom agent**

**Define the functions**

In [3]:
def abc(num1:int, num2:int)->int:
    "Compute abc between two numbers"
    return 2*(num1) - 2*(num2)

In [4]:
abc(2, 3)

-2

In [5]:
def jsonschema(f):
    """
    Generate a JSON schema for the input parameters of the given function.

    Parameters:
        f (FunctionType): The function for which to generate the JSON schema.

    Returns:
        Dict: A dictionary containing the function name, description, and parameters schema.
    """
    kw = {n: (o.annotation, ... if o.default == Parameter.empty else o.default)
            for n, o in inspect.signature(f).parameters.items()}
    s = create_model(f'Input for `{f.__name__}`', **kw).schema()
    return dict(name=f.__name__, description=f.__doc__, parameters=s)

In [6]:
abc_json = jsonschema(abc)
abc_json

/var/folders/s4/cxp4_1c1447576f31nclvwbh0000gp/T/ipykernel_32064/3911407637.py:13: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  s = create_model(f'Input for `{f.__name__}`', **kw).schema()


{'name': 'abc',
 'description': 'Compute abc between two numbers',
 'parameters': {'properties': {'num1': {'title': 'Num1', 'type': 'integer'},
   'num2': {'title': 'Num2', 'type': 'integer'}},
  'required': ['num1', 'num2'],
  'title': 'Input for `abc`',
  'type': 'object'}}

In [7]:
model_name = "qwen/qwen3.8-27b"

**Ask Groq**

In [12]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "compute abc between 2 and 3"},
    ],
)

# print(response.choices[0].message.content)

In [13]:
response.choices[0].message.content

'The phrase "compute abc between 2 and 3" is not standard mathematical notation and is ambiguous. However, I can interpret this in a few likely ways and provide solutions for each:\n\n### Interpretation 1: You meant **a + b + c** (Sum)\nIf you are asking for the sum of numbers between 2 and 3, there are **no integers** strictly between 2 and 3.  \n- If you include the endpoints 2 and 3: $2 + 3 = 5$.  \n- If you are considering real numbers, the integral (continuous sum) of $x$ from 2 to 3 is:\n  $$\n  \\int_{2}^{3} x \\, dx = \\left[ \\frac{x^2}{2} \\right]_2^3 = \\frac{9}{2} - \\frac{4}{2} = \\frac{5}{2} = 2.5\n  $$\n\n### Interpretation 2: You meant **a · b · c** (Product)\nIf you are asking for the product of integers between 2 and 3:\n- There are no integers strictly between 2 and 3.\n- If you include the endpoints: $2 \\times 3 = 6$.\n- If you are considering the definite integral of the product (which doesn\'t make sense without more context), this is less common.\n\n### Interpre

In [ ]:
messages= [
    {"role": "user", "content": "Compute abc between 2 and 3"}
]

# Pass the function to GPT model
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    functions=[abc_json],
    function_call="auto",
    temperature=0
)

In [15]:
response

ChatCompletion(id='chatcmpl-3f1a06a7-458d-487f-812d-7df696041fd7', choices=[Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=FunctionCall(arguments='{"num1":2,"num2":3}', name='abc'), reasoning=None, tool_calls=None))], created=1788234713, model='qwen/qwen3.8-27b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_fea14bdc83', usage=CompletionUsage(completion_tokens=38, prompt_tokens=316, total_tokens=354, completion_time=0.073585648, completion_tokens_details=None, prompt_time=0.021402717, prompt_tokens_details=None, queue_time=0.186426345, total_time=0.094988365), usage_breakdown=None, x_groq=XGroq(id='req_01m1dhgj5beyqtg2wfeh400k5s', debug=None, seed=209214220, usage=None))

**Executing the function by extracting the info from the output of the model**

In [16]:
print(response.choices[0].message.function_call)
print(response.choices[0].message.function_call.arguments)
print(type(response.choices[0].message.function_call.arguments))

FunctionCall(arguments='{"num1":2,"num2":3}', name='abc')
{"num1":2,"num2":3}
<class 'str'>


In [17]:
func_name = response.choices[0].message.function_call.name
func_args = json.loads(response.choices[0].message.function_call.arguments)
print("Function name:", func_name)
print("Function arguments:", func_args)
print(type(func_args))

Function name: abc
Function arguments: {'num1': 2, 'num2': 3}
<class 'dict'>


In [18]:
if func_name == 'abc':
    result = abc(**func_args)
print(result)

-2


## **Using Langchain**

In [ ]:
from langchain_core.tools import tool

@tool
#Coverts the uction into json format
def abc(num1:int, num2:int)->int:
    "Compute abc between two numbers"
    return 2*(num1) - 2*(num2)

In [20]:
abc.description

'Compute abc between two numbers'

In [21]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

In [ ]:
tools = [abc] #conerting to a list
llm_with_tools = llm.bind_tools(tools)

In [25]:
response = llm_with_tools.invoke("Compute abc between 2 and 3")

In [24]:
response.additional_kwargs

{'tool_calls': [{'id': '0dc374x3n',
   'function': {'arguments': '{"num1":2,"num2":3}', 'name': 'abc'},
   'type': 'function'}]}